import pandas as pd
import requests
import datetime
import time

def get_binance_klines(symbol="BTCUSDT", interval="1h", start="2017-01-01"):# skida istorijske Bitcoin cene sa Binance API.
    """
    Download historical OHLCV data from Binance.
    interval examples: 1m, 5m, 15m, 1h, 4h, 1d
    """
    
    url = "https://api.binance.com/api/v3/klines"# endpoint za price data.
    start_ts = int(pd.Timestamp(start).timestamp() * 1000)# pretvara datum u timestamp.
    end_ts = int(time.time() * 1000)

    all_data = []

    while start_ts < end_ts:# Skida podatke u batch-evima jer API ima limit.
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_ts,
            "limit": 1000  # Binance max per request
        }

        data = requests.get(url, params=params).json()

        # Stop if Binance returns empty list (end of available data)
        if not data:
            break

        all_data.extend(data)

        # Move start to last returned timestamp + 1ms
        last_time = data[-1][0]
        start_ts = last_time + 1

        # avoid hitting rate limit
        time.sleep(0.4)

    # Convert into DataFrame
    df = pd.DataFrame(all_data, columns=[
        "OpenTime", "Open", "High", "Low", "Close", "Volume",
        "CloseTime", "QuoteVolume", "Trades", "TakerBuyBase",
        "TakerBuyQuote", "Ignore"
    ])

    # Clean up
    df["OpenTime"] = pd.to_datetime(df["OpenTime"], unit="ms")
    df = df.set_index("OpenTime")

    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float)

    # Rename to match yfinance naming style
    df.index.name = "Datetime"

    return df

df = get_binance_klines("BTCUSDT", interval="1h", start="2017-01-01")# skida Bitcoin cenu po satu od 2017.
df.head()# prikaz prvih redova

In [3]:
#df.to_csv("btc_1h.csv")# cuvanje podataka u csv

/\ /\ /\ Nabavljanje podataka /\ /\ /\

In [4]:
import pandas as pd
df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)# ucitavanje podataka
df.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [5]:
#!pip install ta
import ta

#return
df["return_1h"] = df["Close"].pct_change()# koliko se cena promenila u % zato što model lakše uči promene nego apsolutne cene.
df['daily_return'] = df['Close'].pct_change(24)# Dnevni prinos
#mean/std
df["rolling_mean_24h"] = df["Close"].rolling(24).mean()# prosečna cena zadnja 24h
df["rolling_std_24h"] = df["Close"].rolling(24).std()# volatilnost
#lag
df["close_lag_6h"] = df["Close"].shift(6)# cena pre 6h
df["close_lag_12h"] = df["Close"].shift(12)# cena pre 12h
df["close_lag_24h"] = df["Close"].shift(24)# cena pre 24h
df['close_lag_48h'] = df['Close'].shift(48)  # Cena pre 48h
df['close_lag_168h'] = df['Close'].shift(168)  # Cena pre 7 dana

# RSI
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()  # MACD signal linija
df['MACD_hist'] = macd.macd_diff()  # MACD histogram

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_mavg'] = bb.bollinger_mavg()  # Srednja Bollinger linija
df['BB_upper'] = bb.bollinger_hband()  # Gornja Bollinger linija
df['BB_lower'] = bb.bollinger_lband()  # Donja Bollinger linija

# SMA i EMA
df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
df['EMA_20'] = ta.trend.EMAIndicator(df['Close'], window=20).ema_indicator()

# ATR (Average True Range)
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# ROC (Rate of Change) - Procenat promene cene u poslednjem periodu
df['ROC'] = ta.momentum.ROCIndicator(df['Close'], window=12).roc()

df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,MACD,MACD_signal,MACD_hist,BB_mavg,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,14.272129,26.070185,-11.798056,4158.5800,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,13.525285,23.561205,-10.035920,4155.0285,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,12.892113,21.427387,-8.535274,4150.2985,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,13.836584,19.909226,-6.072642,4146.0650,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,18.142633,19.555907,-1.413274,4145.2810,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352


In [6]:
#dodaje se future return
df["future_close_24h"] = df["Close"].shift(-24)# uzimamo cenu 24h u budućnosti
df["future_return_24h"] = (df["future_close_24h"] - df["Close"]) / df["Close"]# Model ne predviđa cenu nego promenu, to je bolje za ML

def classify_direction(x):
    if x > 0.02:
        return 1      # UP
    elif x < -0.02:
        return -1     # DOWN
    else:
        return 0      # STABLE

#izbacuje NaN
df["direction_24h"] = df["future_return_24h"].apply(classify_direction)

df = df.dropna() #posto je ispod 70% stable, (iako bi mozda trebalo class weights da koristim) za sad cu da ostavim ovako
df["direction_24h"].value_counts(normalize=True)

direction_24h
 0    0.592607
 1    0.217604
-1    0.189789
Name: proportion, dtype: float64

In [7]:
#!pip install scikit-learn

# kad ima mnogo STABLE dana model voli da vara
# Ovo daje veću kaznu za retke klase.

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([-1, 0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["direction_24h"]
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(-1): np.float64(1.756333215506068), np.int64(0): np.float64(0.5624863208579558), np.int64(1): np.float64(1.5318370534796728)}


In [8]:
df.head()

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,BB_mavg,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC,future_close_24h,future_return_24h,direction_24h
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,4158.5800,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456,4310.20,0.047696,1
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,4155.0285,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398,4281.04,0.036047,1
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,4150.2985,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143,4302.72,0.040959,1
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,4146.0650,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481,4329.00,0.042299,1
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,4145.2810,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352,4332.17,0.031469,1


In [9]:
# -----------------------------
# 1) TEMPORAL SPLIT
# -----------------------------


# izdvajamo target
target_col = "direction_24h"

features = [
    "Open","High","Low","Close","Volume",

    "return_1h",
    "daily_return",

    "rolling_mean_24h",
    "rolling_std_24h",

    "close_lag_6h",
    "close_lag_12h",
    "close_lag_24h",
    "close_lag_48h",
    "close_lag_168h",

    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_hist",

    "ATR_14",
    "ROC",

    "SMA_20",
    "EMA_20",
    "BB_upper",
    "BB_lower"
]

# indeks za split (60/20/20)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

#najstariji podaci idu za train, onda za vel i na kraju test
train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(len(train_df), len(val_df), len(test_df))

44718 14906 14906


In [10]:
# -----------------------------
# 2) SCALING
# -----------------------------

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# FIT samo na train
train_df[features] = scaler.fit_transform(train_df[features])

# TRANSFORM na val i test
val_df[features] = scaler.transform(val_df[features])
test_df[features] = scaler.transform(test_df[features])

In [11]:
# -----------------------------
# 3) SEQUENCE CREATION
# -----------------------------

import numpy as np

def create_sequences(data, target, window=168):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(target[i+window-1])# vec je pomereno
    return np.array(X), np.array(y)

window_size = 168

X_train, y_train = create_sequences(
    train_df[features].values,
    train_df[target_col].values,
    window_size
)

X_val, y_val = create_sequences(
    val_df[features].values,
    val_df[target_col].values,
    window_size
)

X_test, y_test = create_sequences(
    test_df[features].values,
    test_df[target_col].values,
    window_size
)

print(X_train.shape, X_val.shape, X_test.shape)

(44550, 168, 24) (14738, 168, 24) (14738, 168, 24)


In [12]:
# -----------------------------
# 4) LABEL FIX (-1,0,1 -> 0,1,2)
# -----------------------------

def relabel(y):
    return np.where(y == -1, 0,
           np.where(y == 0, 1, 2))

y_train = relabel(y_train)
y_val   = relabel(y_val)
y_test  = relabel(y_test)

In [13]:
# -----------------------------
# 5) TORCH DATASET
# -----------------------------
#!pip install torch torchvision torchaudio

import torch
from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val, y_val)
test_dataset  = TimeSeriesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [14]:
# -----------------------------
# 6) PRICE LSTM MODEL
# -----------------------------

import torch.nn as nn

class PriceBranch(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.lstm1 = nn.LSTM(input_size, 128, batch_first=True)
        self.dropout1 = nn.Dropout(0.3)

        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.3)

        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.dropout3 = nn.Dropout(0.3)

        self.fc = nn.Linear(32, 3)  # 3 klase (UP, DOWN, STABLE)

    def forward(self, x):

        out, _ = self.lstm1(x)
        out = self.dropout1(out)

        out, _ = self.lstm2(out)
        out = self.dropout2(out)

        out, _ = self.lstm3(out)
        out = self.dropout3(out)

        out = out[:, -1, :]# Uzima se samo poslednji izlaz iz sekvence
        out = self.fc(out)# Klasifikacija u 3 klase

        return out

In [15]:



# prilagodi class weights za 0,1,2
#weights_tensor = torch.tensor(
#    [class_weights[-1], class_weights[0], class_weights[1]],
#    dtype=torch.float32
#).to(device)

#criterion = nn.CrossEntropyLoss(weight=weights_tensor)# classification loss# presao san na rucno unosenje





#optimizer = torch.optim.Adam(model.parameters(), lr=0.001)# ne moze da bira


In [16]:
# -----------------------------
# 7) TRAINING LOOP
# -----------------------------

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np

#optimizer = get_optimizer(optimizer_type='adam', lr=0.001)# da moze da bira

def training_loop(optimizer_type='adam', lr=0.001, epochs=20):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PriceBranch(input_size=len(features)).to(device)

    if optimizer_type == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_type == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_type == 'rmsprop':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer type: {optimizer_type}")

    weights_tensor = torch.tensor(
        [class_weights[-1], class_weights[0], class_weights[1]],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
##############################################
    # Ručno dodeljene težine za klase
    # Povećaj težinu za manje zastupljene klase (UP i DOWN), smanji za STABLE
    #class_weights = torch.tensor([12, 4.5, 17], dtype=torch.float32).to(device)#prvo je ovo [18, 6, 29.5] onda je [18, 6, 20] bilo bolje

    # Kreiraj funkciju gubitka sa težinama
    #criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
###############################################
    # learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

    # -----------------------------
    # |||| Early stopping setup ||||
    # -----------------------------
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0

    #epochs = 20

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)  # Model predviđa
            loss = criterion(outputs, y_batch)  # Koliko je model pogrešio
            loss.backward()  # Računanje gradijenata
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)  # Stabilizacija treninga (gradient clipping)
            optimizer.step()  # Model uči

            train_loss += loss.item()

        train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_preds = []
        val_true = []
        val_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1).cpu().numpy()# predviđanje klasa za klasifikaciju
                val_preds.extend(preds)
                val_true.extend(y_batch.cpu().numpy())
        val_loss = val_loss / len(val_loader)

        # score evaluacija
        acc = accuracy_score(val_true, val_preds)
        directional_accuracy = np.mean(np.array(val_preds) == np.array(val_true))
        f1 = f1_score(val_true, val_preds, average='weighted')  
        conf_matrix = confusion_matrix(val_true, val_preds)
        print(f"Epoch {epoch+1} | Optimizer : {optimizer_type} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Validation[ Acc: {acc:.4f} | "
            f"Directional Accuracy: {directional_accuracy:.4f} | "
            f"F1: {f1:.4f} ]")
        print(f"\nConfusion Matrix: \n{conf_matrix}\n")
        print("Pred class distribution:", np.bincount(val_preds, minlength=3))
        print("True class distribution:", np.bincount(val_true, minlength=3))

        # Check for early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print("Early stopping...")
            break
        
        # Scheduler za smanjenje learning rate-a
        scheduler.step(val_loss)#scheduler.step(train_loss)
        

In [17]:
training_loop(optimizer_type='adam', lr=0.001, epochs=5)

Epoch 1 | Optimizer : adam | Train Loss: 1.0927 | Val Loss: 1.1041 | Validation[ Acc: 0.2512 | Directional Accuracy: 0.2512 | F1: 0.2616 ]

Confusion Matrix: 
[[1510  309    6]
 [8333 2157   69]
 [1873  446   35]]

Pred class distribution: [11716  2912   110]
True class distribution: [ 1825 10559  2354]
Epoch 2 | Optimizer : adam | Train Loss: 1.0933 | Val Loss: 1.0786 | Validation[ Acc: 0.2686 | Directional Accuracy: 0.2686 | F1: 0.2884 ]

Confusion Matrix: 
[[1340  485    0]
 [7940 2619    0]
 [1724  630    0]]

Pred class distribution: [11004  3734     0]
True class distribution: [ 1825 10559  2354]
Epoch 3 | Optimizer : adam | Train Loss: 1.0905 | Val Loss: 1.0911 | Validation[ Acc: 0.2414 | Directional Accuracy: 0.2414 | F1: 0.2438 ]

Confusion Matrix: 
[[1601  213   11]
 [8611 1934   14]
 [1992  339   23]]

Pred class distribution: [12204  2486    48]
True class distribution: [ 1825 10559  2354]
Epoch 4 | Optimizer : adam | Train Loss: 1.0888 | Val Loss: 1.0731 | Validation[ Acc:

In [18]:
training_loop(optimizer_type='sgd', lr=0.0001, epochs=5)# manji learn rate za sgd

Epoch 1 | Optimizer : sgd | Train Loss: 1.0981 | Val Loss: 1.0696 | Validation[ Acc: 0.7164 | Directional Accuracy: 0.7164 | F1: 0.5981 ]

Confusion Matrix: 
[[    0  1825     0]
 [    0 10559     0]
 [    0  2354     0]]

Pred class distribution: [    0 14738     0]
True class distribution: [ 1825 10559  2354]
Epoch 2 | Optimizer : sgd | Train Loss: 1.0969 | Val Loss: 1.0656 | Validation[ Acc: 0.7164 | Directional Accuracy: 0.7164 | F1: 0.5981 ]

Confusion Matrix: 
[[    0  1825     0]
 [    0 10559     0]
 [    0  2354     0]]

Pred class distribution: [    0 14738     0]
True class distribution: [ 1825 10559  2354]
Epoch 3 | Optimizer : sgd | Train Loss: 1.0964 | Val Loss: 1.0635 | Validation[ Acc: 0.7164 | Directional Accuracy: 0.7164 | F1: 0.5981 ]

Confusion Matrix: 
[[    0  1825     0]
 [    0 10559     0]
 [    0  2354     0]]

Pred class distribution: [    0 14738     0]
True class distribution: [ 1825 10559  2354]
Epoch 4 | Optimizer : sgd | Train Loss: 1.0959 | Val Loss: 1.

In [19]:
training_loop(optimizer_type='rmsprop', lr=0.001, epochs=5)

Epoch 1 | Optimizer : rmsprop | Train Loss: 1.0712 | Val Loss: 1.0078 | Validation[ Acc: 0.7130 | Directional Accuracy: 0.7130 | F1: 0.5989 ]

Confusion Matrix: 
[[   13  1812     0]
 [   64 10495     0]
 [   42  2312     0]]

Pred class distribution: [  119 14619     0]
True class distribution: [ 1825 10559  2354]
Epoch 2 | Optimizer : rmsprop | Train Loss: 1.0690 | Val Loss: 1.0426 | Validation[ Acc: 0.7171 | Directional Accuracy: 0.7171 | F1: 0.6014 ]

Confusion Matrix: 
[[    0  1807    18]
 [    0 10547    12]
 [    0  2333    21]]

Pred class distribution: [    0 14687    51]
True class distribution: [ 1825 10559  2354]
Epoch 3 | Optimizer : rmsprop | Train Loss: 1.0685 | Val Loss: 1.0443 | Validation[ Acc: 0.7171 | Directional Accuracy: 0.7171 | F1: 0.6031 ]

Confusion Matrix: 
[[    0  1800    25]
 [    0 10536    23]
 [    2  2319    33]]

Pred class distribution: [    2 14655    81]
True class distribution: [ 1825 10559  2354]
Epoch 4 | Optimizer : rmsprop | Train Loss: 1.064